# A2：ACC 自适应巡航 —— 跟驰控制

## 实验目标
- 理解 ACC 的跟驰逻辑（间距+速度双闭环）
- 实现 `ACCController`，在 esmini 真闭环中验证

## 通过判据
```python
assert max_spacing_error <= 1.5  # 20s 跟驰无碰撞
```


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import EsminiEnv
print('✓ 导入成功')


## 1. 实现你的 ACC 控制器

In [ ]:
class MyACCController(BaseController):
    """ACC 自适应巡航（学生实现区）"""
    def __init__(self, config=None):
        super().__init__(config)
        self.headway = self.config.get('headway', 1.0)
        self.kp, self.kd = 0.8, 0.5
    def step(self, observation):
        ego, targets = observation['ego'], observation.get('targets', [])
        if not targets:
            return {'target_a': 0.5, 'steer_angle': 0.0}  # 巡航加速
        lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
        # 间距误差 + 速度误差 PD 控制
        gap_err = (abs(lead['x'] - ego['x']) - 4.5) - ego['v'] * self.headway
        v_err = lead['v'] - ego['v']
        a = self.kp * gap_err + self.kd * v_err
        return {'target_a': max(-3.0, min(2.0, a)), 'steer_angle': 0.0}
print('✓ ACC 控制器定义完成')


## 2. 真 esmini 闭环仿真

In [ ]:
controller = MyACCController(config={'headway': 1.0})
env = EsminiEnv(scenario='/home/jovyan/labs/ad2026/cutin.xosc',
                dt=0.05)
obs = env.reset(ego_x=10.0, ego_y=0.0, ego_yaw=0.0, ego_v=15.0)
controller.reset()

max_spacing_error, is_collision = 0.0, False
history = []
for step in range(400):  # 20s
    action = controller.step(obs)
    obs, done, info = env.step(action)
    ego = obs['ego']
    for tgt in obs.get('targets', []):
        gap = abs(tgt['x'] - ego['x']) - 4.5
        err = abs(gap - ego['v'] * 1.0)
        max_spacing_error = max(max_spacing_error, err)
        if gap < 0.5:
            is_collision = True
    history.append((obs['sim_time'], ego['v']))
    if done or is_collision:
        break
env.close()
print(f'✓ 完成：{len(history)} 步，最大间距误差={max_spacing_error:.2f}m，碰撞={is_collision}')


## 3. 评测断言

In [ ]:
assert not is_collision, '发生碰撞!'
assert max_spacing_error <= 1.5, f'间距误差 {max_spacing_error:.2f}m 超标!'
print('✅ 评测通过！')


## 4. 可视化

In [ ]:
import matplotlib.pyplot as plt
times = [t for t, v in history]
speeds = [v for t, v in history]
plt.figure(figsize=(10, 4))
plt.plot(times, speeds)
plt.xlabel('Time (s)')
plt.ylabel('Ego Speed (m/s)')
plt.title('ACC Following - Ego Speed')
plt.grid(True, alpha=0.3)
plt.show()


## 🏆 挑战任务
1. 把 headway 改为 0.8/1.2，对比跟驰稳定性。
2. 在控制器中加入舒适性约束（加速度变化率限制）。
